# Marchenko–Pastur setup

Return matrix \(R \in \mathbb{R}^{n \times m}\), row-centered \(X\), sample covariance
\(C_b = XX^\top / m\). For i.i.d. noise with variance \(\sigma^2\), aspect ratio
\(\gamma = n/m\), bounds \(a = \sigma^2(1-\sqrt{\gamma})^2\), \(b = \sigma^2(1+\sqrt{\gamma})^2\).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

## Symmetric eigenvalues in NumPy

For a real symmetric matrix \(S\), `np.linalg.eigvalsh(S)` returns eigenvalues in **ascending** order.
Use `np.linalg.eigh(S)` when you need eigenvectors (columns are orthonormal).

In [ ]:
def symmetric_spectrum(S: np.ndarray, descending: bool = True) -> np.ndarray:
    """Eigenvalues of real symmetric S; optionally ordered λ1 ≥ … ≥ λn."""
    S = np.asarray(S, dtype=float)
    if S.shape[0] != S.shape[1]:
        raise ValueError("S must be square")
    lam = np.linalg.eigvalsh(S)
    if descending:
        lam = lam[::-1]
    return lam


def symmetric_eigh(S: np.ndarray, descending: bool = True):
    """Eigenvalues and eigenvectors of real symmetric S."""
    S = np.asarray(S, dtype=float)
    lam, V = np.linalg.eigh(S)
    if descending:
        lam = lam[::-1]
        V = V[:, ::-1]
    return lam, V

## Returns, centering, and \(C_b\)

Each **row** of \(R\) is one asset’s time series (length \(m\)). Center each row:
\[
X_{i\cdot} = R_{i\cdot} - \bar{R}_{i\cdot}, \qquad C_b = \frac{1}{m} X X^\top.
\]
Eigenvalues of \(C_b\) are ordered \(\lambda_1 \ge \cdots \ge \lambda_n \ge 0\).

In [ ]:
def center_rows(R: np.ndarray) -> np.ndarray:
    """Subtract row means; R is n×m."""
    R = np.asarray(R, dtype=float)
    return R - R.mean(axis=1, keepdims=True)


def sample_covariance_cb(R: np.ndarray) -> np.ndarray:
    """Cb = X X^T / m with row-centered X."""
    X = center_rows(R)
    n, m = X.shape
    return (X @ X.T) / m


def cb_eigenvalues(R: np.ndarray) -> np.ndarray:
    """λ1 ≥ … ≥ λn for Cb from return matrix R."""
    return symmetric_spectrum(sample_covariance_cb(R), descending=True)

## Marchenko–Pastur density

For i.i.d. entries with variance \(\sigma^2\), \(\gamma = n/m\):
\[
a = \sigma^2(1-\sqrt{\gamma})^2, \quad b = \sigma^2(1+\sqrt{\gamma})^2,
\]
\[
f_{\gamma,\sigma^2}(x) = \frac{\sqrt{(b-x)(x-a)}}{2\pi \gamma \sigma^2 x}\,
\mathbf{1}_{\{a < x < b\}},
\]
with an atom of mass \((1 - 1/\gamma)_+\) at zero when \(\gamma > 1\).

In [ ]:
def mp_bounds(gamma: float, sigma2: float) -> tuple[float, float]:
    """Support (a, b) for the MP bulk (requires gamma <= 1 for standard bulk)."""
    if gamma <= 0:
        raise ValueError("gamma must be positive")
    root = np.sqrt(gamma)
    a = sigma2 * (1.0 - root) ** 2
    b = sigma2 * (1.0 + root) ** 2
    return float(a), float(b)


def mp_density(x: np.ndarray, gamma: float, sigma2: float) -> np.ndarray:
    """Marchenko–Pastur continuous density f_{γ,σ²}(x)."""
    x = np.asarray(x, dtype=float)
    a, b = mp_bounds(gamma, sigma2)
    out = np.zeros_like(x, dtype=float)
    mask = (x > a) & (x < b)
    xm = x[mask]
    out[mask] = np.sqrt((b - xm) * (xm - a)) / (2.0 * np.pi * gamma * sigma2 * xm)
    return out


def mp_point_mass_at_zero(gamma: float) -> float:
    """Atom mass (1 - 1/γ)+ at λ = 0."""
    return float(max(0.0, 1.0 - 1.0 / gamma))

## Example: i.i.d. Gaussian noise returns

Simulate \(R \in \mathbb{R}^{n \times m}\) with i.i.d. \(\mathcal{N}(0,\sigma^2)\) entries (each row centered),
compare empirical eigenvalues of \(C_b\) to the MP curve.

In [ ]:
n, m = 200, 400
gamma = n / m
sigma2 = 1.0
sigma = np.sqrt(sigma2)

R = sigma * np.random.standard_normal((n, m))
Cb = sample_covariance_cb(R)
eigs = symmetric_spectrum(Cb, descending=True)

a, b = mp_bounds(gamma, sigma2)
x_grid = np.linspace(max(a, 1e-6), b, 500)
f_mp = mp_density(x_grid, gamma, sigma2)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(eigs, bins=40, density=True, alpha=0.55, color="steelblue", label="empirical λ(Cb)")
ax.plot(x_grid, f_mp, color="crimson", lw=2, label=r"$f_{\gamma,\sigma^2}(x)$")
ax.axvline(a, color="gray", ls="--", lw=1, label=rf"$a={a:.3f}$, $b={b:.3f}$")
ax.axvline(b, color="gray", ls="--", lw=1)
ax.set_xlabel(r"eigenvalue $\lambda$")
ax.set_ylabel("density")
ax.set_title(rf"$n={n}$, $m={m}$, $\gamma={gamma:.2f}$, $\sigma^2={sigma2}$")
ax.legend()
fig.tight_layout()
plt.show()

print(f"γ = {gamma:.4f}, a = {a:.4f}, b = {b:.4f}")
print(f"MP point mass at 0: {mp_point_mass_at_zero(gamma):.4f} (nonzero only if γ > 1)")
print(f"Top 5 λ: {eigs[:5]}")

## Optional: returns as a pandas DataFrame

Rows = assets, columns = time indices.

In [ ]:
cols = [f"t{j}" for j in range(m)]
idx = [f"asset_{i}" for i in range(n)]
R_df = pd.DataFrame(R, index=idx, columns=cols)
X_df = pd.DataFrame(center_rows(R_df.values), index=idx, columns=cols)
Cb_df = pd.DataFrame(Cb, index=idx, columns=idx)
R_df.head()

## Problem 2 (a) — Wigner matrix and semicircle law

Build a **zero-diagonal Wigner** matrix: independent \(N(0,1)\) entries on the strict upper triangle, mirror to symmetry, scale by \(1/\sqrt{n}\). Empirical eigenvalues should match the semicircle density
\[
\rho(x) = \frac{1}{2\pi}\sqrt{4-x^2}\,\mathbf{1}_{\{|x|\le 2\}}.
\]

In [ ]:
def wigner_matrix(n: int, rng: np.random.Generator | None = None) -> np.ndarray:
    """Zero diagonal; N(0,1) on upper triangle, mirrored, divided by sqrt(n)."""
    rng = np.random.default_rng() if rng is None else rng
    W = np.zeros((n, n), dtype=float)
    i, j = np.triu_indices(n, k=1)
    W[i, j] = rng.standard_normal(i.size)
    W = W + W.T
    return W / np.sqrt(n)


def semicircle_density(x: np.ndarray) -> np.ndarray:
    """ρ(x) = (1/2π) sqrt(4 - x²) on |x| ≤ 2."""
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    mask = np.abs(x) <= 2.0
    out[mask] = np.sqrt(4.0 - x[mask] ** 2) / (2.0 * np.pi)
    return out

In [ ]:
n_w = 500
x_sc = np.linspace(-2.0, 2.0, 600)
rho_sc = semicircle_density(x_sc)

rng1 = np.random.default_rng(2026)
rng2 = np.random.default_rng(2027)
mats = [wigner_matrix(n_w, rng1), wigner_matrix(n_w, rng2)]
eig_lists = [np.linalg.eigvalsh(M) for M in mats]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, eigs, k in zip(axes, eig_lists, (1, 2)):
    ax.hist(eigs, bins=45, density=True, alpha=0.55, color="steelblue", label=r"$\lambda(W)$")
    ax.plot(x_sc, rho_sc, color="crimson", lw=2, label=r"$\rho(x)$ semicircle")
    ax.set_xlim(-2.5, 2.5)
    ax.set_xlabel(r"eigenvalue $\lambda$")
    ax.set_title(rf"sample {k}, $n={n_w}$")
    ax.legend(loc="upper right", fontsize=9)
axes[0].set_ylabel("density")
fig.suptitle("Problem 2(a): Wigner eigenvalues vs semicircle law", y=1.02)
fig.tight_layout()
plt.show()

## Sample covariance \(S = \frac{1}{m}XX^\top\) and Marchenko–Pastur (\(\gamma > 1\))

Draw \(X \in \mathbb{R}^{n \times m}\) with i.i.d. \(N(0,1)\) entries (variance \(1\)), set \(S = XX^\top/m\),
and compare the empirical spectrum to MP with \(\gamma = n/m\). When \(n > m\), rank \(\le m\) forces
\(n-m\) **exact** zero eigenvalues; MP adds a point mass \((1 - 1/\gamma)_+\) at \(0\).

**Why not the semicircle?** The semicircle law describes **Wigner** matrices (symmetric, centered independent
entries, typically \(1/\sqrt{n}\) scaling). Here \(S\) is a **Gram / sample covariance** matrix: entries are
sums of \(m\) products, highly dependent, and eigenvalues are **nonnegative** with bulk on \([a,b]\), not on
\([-2,2]\). The correct limit is **Marchenko–Pastur**, not Wigner.

In [ ]:
def gram_covariance(X: np.ndarray) -> np.ndarray:
    """S = (1/m) X X^T for X of shape n×m."""
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    return (X @ X.T) / m


def count_zero_eigenvalues(eigs: np.ndarray, tol: float = 1e-10) -> int:
    return int(np.sum(np.abs(eigs) <= tol))

In [ ]:
n, m = 400, 200
gamma = n / m
sigma2 = 1.0  # entry variance

rng = np.random.default_rng(42)
X = rng.standard_normal((n, m))
S = gram_covariance(X)
eigs = np.linalg.eigvalsh(S)  # ascending

a, b = mp_bounds(gamma, sigma2)
x_grid = np.linspace(max(a, 1e-8), b, 500)
f_bulk = mp_density(x_grid, gamma, sigma2)
# Full MP measure: atom (1 - 1/γ)+ at 0; bulk has weight 1/γ with density f on [a, b].
f_on_esd = f_bulk / gamma

n_zero = count_zero_eigenvalues(eigs)
n_pos = n - n_zero
rank_X = np.linalg.matrix_rank(X)
atom_mass = mp_point_mass_at_zero(gamma)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.hist(
    eigs,
    bins=50,
    density=True,
    alpha=0.55,
    color="steelblue",
    label=rf"empirical ($n={n}$ eigenvalues)",
)
ax.plot(
    x_grid,
    f_on_esd,
    color="crimson",
    lw=2,
    label=rf"MP bulk scaled by $1/\gamma$ ($\sigma^2={sigma2}$)",
)
ax.axvline(0.0, color="black", ls=":", lw=1, alpha=0.6)
ax.axvline(a, color="gray", ls="--", lw=1.5, label=rf"$a={a:.3f}$")
ax.axvline(b, color="gray", ls="--", lw=1.5, label=rf"$b={b:.3f}$")
ax.set_xlim(-0.05, b * 1.05)
ax.set_xlabel(r"eigenvalue $\lambda$ of $S=\frac{1}{m}XX^\top$")
ax.set_ylabel("density (ESD histogram)")
ax.set_title(rf"$n={n}$, $m={m}$, $\gamma={gamma:.2f}$ — MP vs semicircle is wrong model")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"γ = n/m = {gamma:.4f}")
print(f"MP edges: a = {a:.6f}, b = {b:.6f}")
print(f"Theoretical atom at 0: (1 - 1/γ)+ = {atom_mass:.4f}")
print(f"Expected exact zeros (rank ≤ m): n - m = {n - m}")
print(f"Numerical rank of X: {rank_X}")
print(f"Eigenvalues with |λ| ≤ 1e-10: {n_zero} (positive bulk count: {n_pos})")
print(f"Empirical fraction at zero: {n_zero / n:.4f}")
print(f"Largest bulk eigenvalue: {eigs[eigs > 1e-10].max():.4f} (≤ b in limit)")

### Zero eigenvalues

Since \(\operatorname{rank}(XX^\top) \le \min(n,m) = m\), at most \(m\) eigenvalues of \(S\) can be positive.
The remaining \(n - m = 200\) are **exactly zero** (up to floating point). In the MP limit with \(\gamma = 2\),
half of the spectral mass sits at \(0\) (atom \(1 - 1/\gamma = 1/2\)), matching \(200/400\) eigenvalues.

### Semicircle vs MP (short)

| | Wigner (semicircle) | \(S = XX^\top/m\) (MP) |
|---|---|---|
| Structure | symmetric, ~independent entries | Gram matrix, rank \(\le m\) |
| Scaling | \(O(1/\sqrt{n})\) | \(O(1)\) eigenvalues |
| Support | \([-2,2]\) (symmetric) | \([0,b]\) with possible atom at \(0\) |

Using \(\rho(x) = \frac{1}{2\pi}\sqrt{4-x^2}\) would predict negative eigenvalues and symmetric bulk around \(0\);
that does not match nonnegative, skewed Gram spectra.

## Correcting row–variable / column–date setup (\(\sigma^2 = 4\))

**Interpretation:** \(R \in \mathbb{R}^{n \times m}\), row \(i\) = one variable, column \(j\) = one date;
i.i.d. entries with variance \(4\). Sample covariance \(C_b = XX^\top/m\) with **row-centered** \(X\).

| Line | Issue | Fix |
|------|--------|-----|
| `X = R - R.mean(axis=0, ...)` | `axis=0` averages over **rows** (per date); need per-**variable** means | `axis=1, keepdims=True` |
| `S = X.T @ X / n` | \(X^\top X\) is \(m \times m\) (dates); wrong divisor | `S = X @ X.T / m` → \(n \times n\) |
| `eigvals(S) / sqrt(n)` | Wigner scaling; \(S\) is symmetric → `eigvalsh`; MP eigenvalues are \(O(1)\) | `np.linalg.eigvalsh(S)` |
| `gamma = m / n` | MP aspect ratio is \(n/m\) (limits use \(\gamma = n/m\)) | `gamma = n / m` |
| `a, b = (1 ± √γ)²` | Omits entry variance \(\sigma^2 = 4\) | multiply by `sigma2 = 4` |

**Corrected:** \(\dim(S) = n \times n = 120 \times 120\), \(\gamma = n/m = 120/240 = 0.5\).

In [ ]:
rng = np.random.default_rng(0)
n, m = 120, 240
sigma2 = 4.0

R = np.sqrt(sigma2) * rng.standard_normal(size=(n, m))  # same as 2 * normal, var = 4
X = R - R.mean(axis=1, keepdims=True)
S = X @ X.T / m
eig = np.linalg.eigvalsh(S)
gamma = n / m
a, b = sigma2 * (1 - np.sqrt(gamma)) ** 2, sigma2 * (1 + np.sqrt(gamma)) ** 2

print(f"S.shape = {S.shape}, γ = {gamma:.4f}, a = {a:.4f}, b = {b:.4f}")